# AI & Computational Science Applications

**Morning session · 11:05 – 11:55 AM** · [website version](https://training.nrp-nautilus.io/pearc26/3_ai_apps.html) — run cells with **Shift+Enter**.


## ⚙️ Setup — run this first

Set your short username once; every command below uses `$NRP_USER`. The cell
also renders every manifest into **`my-yamls/`** with `<username>` already
filled in — wherever the website says *"replace `<username>`"*, it's already
done for you here.

> Terminal steps below don't share this variable — run the same
> `export NRP_USER=...` line in any terminal you open.
> Re-running this cell re-renders `my-yamls/` (overwriting any edits you made there).


In [ ]:
export NRP_USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/pearc26/workspace
if [ "$NRP_USER" = changeme ]; then echo "⚠️  Edit NRP_USER above first, then re-run"; else
  mkdir -p my-yamls
  for f in yamls/*; do sed "s/<username>/$NRP_USER/g" "$f" > "my-yamls/$(basename "$f")"; done
  echo "✅ my-yamls/ rendered for $NRP_USER"
fi


**Morning session · 11:05 – 11:55 AM**

NRP runs hundreds of NVIDIA GPUs (and Qualcomm Cloud AI 100 cards) across the country. A subset of those GPUs power a community-shared, **OpenAI-compatible LLM inference endpoint** at `https://ellm.nrp-nautilus.io/v1`. This episode walks the full path: talk to the managed LLM from Jupyter AI, `curl`, and Python; then bring up your own GPU pod for training, self-hosted inference, and a RAG pipeline against NRP's managed [Milvus](https://milvus.io) vector database.

> 📘 **Docs:** [Managed LLMs](https://nrp.ai/documentation/userdocs/ai/llm-managed/) · [Available models](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) · [LLM API access](https://nrp.ai/documentation/userdocs/ai/llm-managed/api-access/) · [Vector DB (Milvus)](https://nrp.ai/documentation/userdocs/vector-database/) · [LLM token](https://nrp.ai/llmtoken) · [GPU pods](https://nrp.ai/documentation/userdocs/running/gpu-pods/)


## 1. NRP GPUs power a managed LLM service

NRP exposes GPUs in two complementary ways:

1. **Bring-your-own pod** — request `nvidia.com/gpu` (or model-specific keys) in your container's resources, as in Episode 2. Full control over weights, runtime, and versions.
2. **Managed LLM service** — a rotating catalog of open-weights LLMs hosted on those same GPUs behind the OpenAI-compatible URL `https://ellm.nrp-nautilus.io/v1`. No pod to run, no GPU time to hold; just HTTP requests with a bearer token from [nrp.ai/llmtoken](https://nrp.ai/llmtoken).

See the [models page](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) for the live catalog — large mixture-of-experts chat models, code models, vision-language models, and an embeddings model, all behind one endpoint.

**Browser entry points** (no token needed; sign in with your NRP account):

- [nrp-openwebui.nrp-nautilus.io](https://nrp-openwebui.nrp-nautilus.io) — Open WebUI
- [librechat.nrp-nautilus.io](https://librechat.nrp-nautilus.io) — LibreChat

**💡 Tutorial token & endpoint**

Inside the tutorial JupyterHub, `OPENAI_API_BASE` and `OPENAI_API_KEY` are **already exported** in every terminal and notebook, and injected into pods that mount the `nrp-llm-token` Secret in `nrp-training-k8s`. The examples below use those variables verbatim. After the tutorial, mint your own token at [nrp.ai/llmtoken](https://nrp.ai/llmtoken).


## 2. Talk to the LLM from Jupyter AI

The tutorial hub ships with [Jupyter AI](https://jupyter-ai.readthedocs.io/) **pre-configured for the NRP managed LLM** — nothing to install.

**Try it now — chat panel.** Click the **chat (robot) icon** in the JupyterLab left sidebar, type a question, and send. Replies stream back from a model running on NRP GPUs.

**Or use the cell magic** in a Python 3 notebook:

*(In a Python 3 notebook, not here:)*

```python
%load_ext jupyter_ai_magics
```


```text
%%ai openai-chat:minimax-m2
What is the National Research Platform in two sentences?
```


Switch model per cell — the first line of the magic is `%%ai <provider>:<model>`. `%ai list` shows every registered provider.

**What you learn.** Jupyter AI is the lowest-friction way to demo the managed LLM during a class — students log in, no token handoff, no `pip install`.


## 3. Talk to the LLM with `curl`

The endpoint is OpenAI-compatible — anything that speaks OpenAI's REST API speaks NRP. Open a JupyterLab terminal.

**List models:**


In [ ]:
curl -s -H "Authorization: Bearer $OPENAI_API_KEY" \
     "$OPENAI_API_BASE/models" | python3 -m json.tool | head -30


**Send a chat completion:**


In [ ]:
curl -s -X POST "$OPENAI_API_BASE/chat/completions" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "minimax-m2",
    "messages": [
      {"role": "system", "content": "Answer in one sentence."},
      {"role": "user",   "content": "What is the National Research Platform?"}
    ]
  }' | python3 -c 'import json,sys; print(json.load(sys.stdin)["choices"][0]["message"]["content"])'


**Stream tokens** (add `"stream": true` and watch SSE chunks arrive):


In [ ]:
curl -sN -X POST "$OPENAI_API_BASE/chat/completions" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{"model":"minimax-m2","stream":true,"messages":[{"role":"user","content":"Count 1 to 5 with a brief reason for each."}]}'


**What you learn.** `curl` is the universal smoke test — if it works here, anything OpenAI-compatible (LangChain, openai-python, your own app) works.


## 4. Talk to the LLM with Python (`openai` SDK)

The client is pre-installed on hub spawns (`pip install openai` elsewhere):


In [ ]:
python3 <<'PYEOF'
import os
from openai import OpenAI

client = OpenAI()   # reads OPENAI_API_KEY / OPENAI_API_BASE from the environment

resp = client.chat.completions.create(
    model="minimax-m2",
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant."},
        {"role": "user",   "content": "Explain Kubernetes namespaces in two sentences."},
    ],
)
print(resp.choices[0].message.content)
PYEOF


**Streaming:**


In [ ]:
python3 <<'PYEOF'
from openai import OpenAI
client = OpenAI()
stream = client.chat.completions.create(
    model="minimax-m2",
    messages=[{"role": "user", "content": "Write a haiku about GPUs."}],
    stream=True,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()
PYEOF


**What you learn.** The exact same code targets the OpenAI cloud, NRP's managed LLM, or any vLLM/TGI server you bring up yourself — only `base_url` changes. This portability is the entire point of the OpenAI-compatible API.


## 5. Bring your own GPU pod

The managed LLM is convenient but constrained: you don't pick the weights, version, quantization, or runtime. When you need that control — or want to run training — request a GPU yourself. Every manifest below already includes the tutorial reservation pattern from Episode 2 (toleration + A10 affinity).


### 5.1 PyTorch GPU sanity check + 1-epoch MNIST

`yamls/pytorch-training.yaml` requests **1 NVIDIA A10**, runs `nvidia-smi`, then trains MNIST for one epoch. Apply (replace `<username>`):


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/pytorch-training.yaml


**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl get pod -n nrp-training-k8s tutorial-$NRP_USER-gp3 -w
```


Once `Completed`:


In [ ]:
kubectl logs -n nrp-training-k8s tutorial-$NRP_USER-gp3 | tail -25


<details>
<summary>Expected output (truncated)</summary>

<pre style="line-height:1.45">
|   0  NVIDIA A10                  ...    |  ...                 |   0%      Default    |

Train Epoch: 1 [0/60000 (0%)]    Loss: 2.305199
...
Test set: Average loss: 0.0501, Accuracy: 9849/10000 (98%)
PyTorch MNIST completed successfully.
</pre>
</details>


Cleanup:


In [ ]:
kubectl delete -n nrp-training-k8s -f my-yamls/pytorch-training.yaml


### 5.2 Run your own LLM with TGI

Same pattern with HuggingFace's [Text Generation Inference](https://github.com/huggingface/text-generation-inference) serving `HuggingFaceH4/zephyr-7b-beta` on a single A10:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/tgi-inference.yaml


**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl get pod -n nrp-training-k8s tutorial-$NRP_USER-tgi -w
```


Wait 1–3 minutes for the model download, then port-forward and query it:

**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl port-forward -n nrp-training-k8s tutorial-$NRP_USER-tgi 8080:80
```


In [ ]:
# second terminal:
curl -s http://127.0.0.1:8080/generate \
  -H "Content-Type: application/json" \
  -d '{"inputs":"Why are penguins black and white?","parameters":{"max_new_tokens":60}}'


And the punchline — TGI exposes an OpenAI-compatible `/v1`, so the §4 code works against **your own GPU** by changing only the base URL:


In [ ]:
python3 <<'PYEOF'
from openai import OpenAI
client = OpenAI(api_key="not-needed", base_url="http://127.0.0.1:8080/v1")
print(client.chat.completions.create(
    model="tgi",
    messages=[{"role":"user","content":"Hi from my own GPU."}],
).choices[0].message.content)
PYEOF


Cleanup:


In [ ]:
kubectl delete -n nrp-training-k8s -f my-yamls/tgi-inference.yaml


### 5.3 RAG over the NRP docs with Milvus

The densest exercise of the morning: a managed vector database, the managed LLM, and a retrieval pipeline wired together. A pod (a) clones the public NRP docs, (b) chunks and embeds every page with `sentence-transformers/all-MiniLM-L6-v2`, (c) writes vectors to the managed **Milvus** cluster at `milvus.nrp-nautilus.io:50051`, and (d) answers questions by retrieving the top-4 chunks and sending them to the LLM with the system prompt *"answer only from this context — if it isn't there, say so."*

The pod mounts two pre-loaded Secrets from `nrp-training-k8s`:

| Secret | Keys | Source |
|---|---|---|
| `nrp-training-milvus-credentials` | `host`, `port`, `username`, `password`, `secure`, `database` | [nrp.ai/milvus](https://nrp.ai/milvus) |
| `nrp-llm-token` | `OPENAI_API_BASE`, `OPENAI_API_KEY` | [nrp.ai/llmtoken](https://nrp.ai/llmtoken) |

**Stage 1 — bring up the RAG pod** (bootstrap takes 3–5 min after `Running`):


In [ ]:
kubectl apply  -n nrp-training-k8s -f my-yamls/milvus-rag.yaml


**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl get pod -n nrp-training-k8s tutorial-$NRP_USER-vectordb -w
```


**Stage 2 — build the index** (the workspace ships `yamls/nrp_docs_rag.py`, one ~290-line script — read it; nothing magic):


In [ ]:
kubectl cp my-yamls/nrp_docs_rag.py nrp-training-k8s/tutorial-$NRP_USER-vectordb:/scratch/


**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl exec -it -n nrp-training-k8s tutorial-$NRP_USER-vectordb -- bash
```


In [ ]:
cd /scratch
python3 nrp_docs_rag.py --reindex


Chunking is instant, embedding ~960 chunks takes ~25 s on the A10, and the collection **persists in Milvus across runs** — later invocations answer in seconds.

**Stage 3 — ask questions:**


In [ ]:
python3 nrp_docs_rag.py --only-ask \
  --ask "How do I get a Milvus database password on NRP, and what is the connection endpoint?"


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
Q: How do I get a Milvus database password on NRP, and what is the connection endpoint?
------------------------------------------------------------------------------
To get your Milvus database password, navigate to the Milvus password page
(/milvus) and click the &quot;Get milvus password&quot; button; a link to a secure page
containing your password will be sent to your email.

The Milvus GRPC endpoint is **milvus.nrp-nautilus.io:50051**.

Source: https://nrp.ai/documentation/userdocs/ai/vector-database
</pre>
</details>


Also try a question whose answer is **not** in the docs — a well-grounded pipeline should decline rather than confabulate:


In [ ]:
python3 nrp_docs_rag.py --only-ask \
  --ask "What does the cluster do if a pod has no CPU or memory limits?"


**What you learn.** Same code, same retriever, same prompt — the inference backend is swappable (managed endpoint, your own TGI pod, a local Ollama). Pick per deployment context: cost, latency, privacy.

Cleanup:


In [ ]:
kubectl delete -n nrp-training-k8s -f my-yamls/milvus-rag.yaml


The Milvus collection survives the pod — the next RAG pod reuses it without re-indexing.


## 6. End-of-morning cleanup


In [ ]:
kubectl delete -n nrp-training-k8s -f my-yamls/pytorch-training.yaml --ignore-not-found
kubectl delete -n nrp-training-k8s -f my-yamls/tgi-inference.yaml    --ignore-not-found
kubectl delete -n nrp-training-k8s -f my-yamls/milvus-rag.yaml       --ignore-not-found


Stop any `kubectl port-forward` processes, then verify with `bash check.sh 3`.


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.


In [ ]:
bash check.sh 3
